In [0]:
%sql
CREATE CATALOG IF NOT EXISTS mini_project

In [0]:
# # environment variables to connect to ngrok hosted postgresql
# host = "6.tcp.ngrok.io"
# port = 15276
# db   = "Adventureworks"
# user = "postgres"
# pwd  = "postgres"

# jdbc_url = f"jdbc:postgresql://{host}:{port}/{db}?sslmode=disable"

In [0]:
# # credential payload to submit to the jdbc driver
# props = {
#   "user": user,
#   "password": pwd,
#   "driver": "org.postgresql.Driver"
# }

In [0]:
# Where to land tables in raw layer (Unity Catalog)
CATALOG = "mini_project"
BRONZE_SCHEMA  = "bronze_layer"   # change if you want
POSTGRES_CATALOG = 'postgres_adventureworks'

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")


In [0]:
SCHEMA_INCLUSION_LIST = [
    'humanresources',
    'person',
    'production',
    'purchasing',
    'sales'
]

In [0]:
from pyspark.sql import functions as F

# Get all schemas + tables from Postgres catalog
tables_df = spark.sql(f"""
    SELECT table_schema, table_name
    FROM {POSTGRES_CATALOG}.information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema IN ({', '.join([f"'{schema}'" for schema in SCHEMA_INCLUSION_LIST])})
""")

tables = tables_df.collect()

for row in tables:
    src_schema = row["table_schema"]
    table_name = row["table_name"]

    source_table = f"{POSTGRES_CATALOG}.{src_schema}.{table_name}"
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{src_schema}_{table_name}"

    print(f"Loading {source_table} -> {target_table}")

    df = spark.read.table(source_table)

    (
        df
        .write
        .mode("overwrite")   # use append later for incrementals
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

In [0]:
# SCHEMA_TABLE_LIST = [
#   ("humanresources","department"),
#   ("humanresources","employee"),
#   ("humanresources","employeedepartmenthistory"),
#   ("humanresources","employeepayhistory"),
#   ("humanresources","jobcandidate"),
#   ("humanresources","shift"),
#   ("person","address"),
#   ("person","addresstype"),
#   ("person","businessentity"),
#   ("person","businessentityaddress"),
#   ("person","businessentitycontact"),
#   ("person","contacttype"),
#   ("person","countryregion"),
#   ("person","emailaddress"),
#   ("person","password"),
#   ("person","person"),
#   ("person","personphone"),
#   ("person","phonenumbertype"),
#   ("person","stateprovince"),

#   ("production","billofmaterials"),
#   ("production","culture"),
#   ("production","document"),
#   ("production","illustration"),
#   ("production","location"),
#   ("production","product"),
#   ("production","productcategory"),
#   ("production","productcosthistory"),
#   ("production","productdescription"),
#   ("production","productdocument"),
#   ("production","productinventory"),
#   ("production","productlistpricehistory"),
#   ("production","productmodel"),
#   ("production","productmodelillustration"),
#   ("production","productmodelproductdescriptionculture"),
#   ("production","productphoto"),
#   ("production","productproductphoto"),
#   ("production","productreview"),
#   ("production","productsubcategory"),
#   ("production","scrapreason"),
#   ("production","transactionhistory"),
#   ("production","transactionhistoryarchive"),
#   ("production","unitmeasure"),
#   ("production","workorder"),
#   ("production","workorderrouting"),

#   ("purchasing","productvendor"),
#   ("purchasing","purchaseorderdetail"),
#   ("purchasing","purchaseorderheader"),
#   ("purchasing","shipmethod"),
#   ("purchasing","vendor"),

#   ("sales","countryregioncurrency"),
#   ("sales","creditcard"),
#   ("sales","currency"),
#   ("sales","currencyrate"),
#   ("sales","customer"),
#   ("sales","personcreditcard"),
#   ("sales","salesorderdetail"),
#   ("sales","salesorderheader"),
#   ("sales","salesorderheadersalesreason"),
#   ("sales","salesperson"),
#   ("sales","salespersonquotahistory"),
#   ("sales","salesreason"),
#   ("sales","salestaxrate"),
#   ("sales","salesterritory"),
#   ("sales","salesterritoryhistory"),
#   ("sales","shoppingcartitem"),
#   ("sales","specialoffer"),
#   ("sales","specialofferproduct"),
#   ("sales","store"),
# ]


In [0]:
# results = []

# for sch, tbl in SCHEMA_TABLE_LIST:
#   source = f'{sch}."{tbl}"'  # quote table names for safety
#   target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{sch}_{tbl}".lower()

#   try:
#     df = (spark.read
#           .format("jdbc")
#           .option("url", jdbc_url)
#           .option("dbtable", source)
#           .options(**props)
#           .load())

#     (df.write
#        .mode("overwrite")
#        .format("delta")
#        .option("overwriteSchema", "true")
#        .saveAsTable(target_table))

#     results.append((sch, tbl, df.count(), "OK"))
#     print(f" Loaded {source} -> {target_table}")

#   except Exception as e:
#     results.append((sch, tbl, None, f"FAIL: {str(e)[:300]}"))
#     print(f" FAILED {source}: {str(e)[:300]}")


In [0]:
# Ingestion status
summary_df = spark.createDataFrame(results, ["schema","table","row_count","status"])
display(summary_df.orderBy("schema","table"))

In [0]:
%sql
select * from hackathon.mini_project_raw.sales_currency